[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance1_cours.ipynb)

# Séance 3.1 — Décrire une distribution

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qu'une moyenne décrit — et ce qu'elle ne décrit pas
- choisir entre moyenne et médiane selon la forme de la distribution
- lire un `describe()` ligne par ligne
- mesurer la dispersion avec l'écart-type et l'écart interquartile
- repérer une concentration : quelle part du total tient dans le haut du classement
- distinguer une moyenne pondérée d'une moyenne non pondérée

## Une phrase de rapport, et le problème

> *« Le panier moyen de nos clients est de 590 €. »*

Cette phrase se trouve dans à peu près tous les rapports d'activité. Elle a
l'air d'une information. Posez-vous la question suivante :

**Si vous deviez fixer le seuil de livraison gratuite, le mettriez-vous à
590 € ?**

À la fin de cette séance vous saurez pourquoi la réponse est non, et ce qu'il
fallait regarder à la place.

### Les données

Même détaillant que le bloc 2, mais à une **maille** différente : une ligne
n'est plus un produit vendu, c'est **une commande entière**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")   ## une ligne = une COMMANDE

print(cmd.shape)   ## 1 955 commandes, et non 45 123 lignes de vente
cmd.head(3)

Trois grandeurs à ne pas confondre :

| Colonne | Ce que c'est |
|---|---|
| `ca` | le montant total de la commande, en euros |
| `nart` | le nombre de **produits distincts** qu'elle contient |
| `qte` | le nombre total d'**unités** commandées, toutes lignes confondues |

Une commande de 3 produits distincts en 100 exemplaires chacun a donc
`nart` = 3 et `qte` = 300.

## 1. La moyenne n'est pas le milieu

Deux façons de résumer une variable par un seul nombre :

- la **moyenne** : on additionne les valeurs de la variable sur toutes les
  observations, puis on divise par le **nombre d'observations**. Ici : la
  somme des 1 955 montants de commande, divisée par 1 955.
- la **médiane** : on classe les observations par valeur croissante et on
  retient celle qui occupe le **milieu du classement**. La moitié des
  observations est en dessous, l'autre moitié au-dessus.

Le vocabulaire compte : une moyenne se calcule *sur une variable*, *pour un
ensemble d'observations*. Dire « la moyenne des commandes » sans préciser
quelle grandeur on additionne ne veut rien dire.

In [ ]:
# somme des montants / nombre de commandes
print("moyenne :", round(cmd["ca"].mean(), 2))
# le montant qui coupe les 1 955 commandes classees en deux moities
print("mediane :", round(cmd["ca"].median(), 2))

**590 € contre 356 €.** Un écart de 234 €, soit deux tiers de la médiane. Ces
deux nombres décrivent le même fichier.

Question avant d'exécuter la cellule suivante : à votre avis, quelle
proportion des commandes dépasse la « moyenne » ?

In [ ]:
# (colonne > valeur) donne des True/False ; leur moyenne est une proportion
part = 100 * (cmd["ca"] > cmd["ca"].mean()).mean()

print(round(part, 1), "% des commandes depassent la moyenne")

**29 %.** Sept commandes sur dix sont en dessous de la « moyenne ». Un seuil
de livraison gratuite à 590 € serait hors de portée pour 71 % des commandes.

Regardons pourquoi.

In [ ]:
cmd["ca"].plot(kind="hist", bins=40, figsize=(7, 4))   ## la FORME
plt.title("Repartition de ca")
plt.xlabel("ca : montant de la commande (euros)")
plt.show()

Voilà la forme : un **tassement à gauche** et une **traîne qui s'étire loin à
droite**. Quelques commandes énormes (jusqu'à 16 775 €) tirent la moyenne vers
le haut sans déplacer la médiane d'un centime.

> 💡 **La règle.** Distribution symétrique : moyenne et médiane coïncident,
> prenez l'une ou l'autre. Distribution asymétrique — et **presque tout ce qui
> est en euros l'est** — la médiane décrit la situation typique, la moyenne
> décrit le total divisé par l'effectif. Affichez les deux.

## 2. Tout voir d'un coup : `describe()`

In [ ]:
cmd["ca"].describe().round(2)   ## huit nombres, une distribution

Huit nombres, et la distribution est décrite :

| Ligne | Ce que ça dit ici |
|---|---|
| `count` | 1 955 commandes |
| `mean` | 589,73 € — la moyenne |
| `std` | 918,41 € — l'écart-type, voir plus bas |
| `min` | 1,45 € — la plus petite commande |
| `25%` | 189,65 € — un quart des commandes sont en dessous |
| `50%` | 355,89 € — la médiane |
| `75%` | 659,52 € — trois quarts sont en dessous |
| `max` | 16 774,72 € — la plus grosse |

### Les quantiles répondent aux questions de seuil

« À partir de quel montant une commande fait-elle partie des 10 % les plus
grosses ? » se lit directement :

In [ ]:
# quantile(0.9) : 90 % des commandes sont EN DESSOUS de ce montant
print("seuil des 10 % du haut :", round(cmd["ca"].quantile(0.9), 2))
print("seuil des 10 % du bas  :", round(cmd["ca"].quantile(0.1), 2))

> ⚠️ **Le piège des quantiles :** ils s'expriment en **proportion**, entre 0
> et 1, jamais en pourcentage. La cellule suivante est volontairement fausse.

In [ ]:
cmd["ca"].quantile(90)   ## erreur volontaire : 90 au lieu de 0.9

Dernière ligne :

```
ValueError: percentiles should all be in the interval [0, 1]
```

Traduction : on attend une proportion entre 0 et 1. Les 10 % du haut, c'est
`quantile(0.9)`, pas `quantile(90)`. **Seule la dernière ligne d'une erreur
compte.**

### Mesurer la dispersion

Deux commandes à 350 € et 360 €, ou deux commandes à 10 € et 700 € : même
moyenne, situation très différente. La **dispersion** mesure cet étalement.

In [ ]:
q1 = cmd["ca"].quantile(0.25)   ## un quart des commandes en dessous
q3 = cmd["ca"].quantile(0.75)   ## trois quarts en dessous

print("ecart-type          :", round(cmd["ca"].std(), 2))   ## fragile
print("ecart interquartile :", round(q3 - q1, 2))           ## robuste

L'**écart-type** (`std`) mesure l'écart typique des observations à leur
moyenne. Il se lit dans la même unité que la variable — ici des euros.

**Regardez-le bien : 918 €, soit davantage que la moyenne elle-même.** Sur une
variable positive comme un montant, un écart-type supérieur à la moyenne est
un **indicateur** de dispersion très forte. C'est un signe, pas une preuve :
590 € reste la moyenne des 1 955 commandes, et elle garde son sens — c'est le
chiffre d'affaires total rapporté au nombre de commandes.

Ce que cet indicateur suggère, c'est autre chose : à ce niveau de dispersion,
la moyenne renseigne mal sur ce que dépense **une** commande prise au hasard.
La bonne réaction n'est pas de renoncer à la moyenne, c'est de ne pas la citer
seule — médiane et quantiles à côté.

L'**écart interquartile** (q3 − q1, ici 470 €) est sa version robuste : il
décrit l'étalement de la moitié centrale et ignore les extrêmes. Il ne bouge
pas si la plus grosse commande double.

## 3. Où est concentré le chiffre d'affaires ?

Une traîne à droite pose toujours la même question business : **quelle part du
total tient dans le haut du classement ?**

In [ ]:
top = cmd["ca"].sort_values(ascending=False)   ## les plus grosses d'abord
n10 = int(0.10 * len(cmd))   ## les 10 % de commandes les plus grosses

part = 100 * top.head(n10).sum() / top.sum()   ## leur poids dans le total
print(n10, "commandes font", round(part, 1), "% du chiffre d'affaires")

**195 commandes sur 1 955 font 41,3 % du chiffre d'affaires.**

Vous avez déjà croisé ce phénomène en séance 2.3 : deux clients irlandais
pesaient 22,7 % du CA. Ce n'était pas une anomalie isolée, c'est la façon dont
ce marché est fait.

Une moyenne **ne contient pas** cette information. Ce n'est pas qu'elle
l'effacerait ou la dissimulerait : elle répond à une autre question, « le total
rapporté à l'effectif », et la répartition du total n'entre nulle part dans ce
calcul. D'où la nécessité de calculer la concentration séparément — c'est une
question distincte, elle demande un chiffre distinct.

## 4. Comparer des groupes sans se faire piéger

Même question, pays par pays. Notez l'ordre des colonnes : **`count` d'abord**.

In [ ]:
# count EN PREMIER : on ne commente pas un groupe sans son effectif
parpays = cmd.groupby("pays")["ca"].agg(["count", "mean", "median"])

parpays.query("count >= 20").sort_values("mean", ascending=False).round(2)

Trois lectures :

- **La Suède : 1 409 € de moyenne, 486 € de médiane.** L'écart le plus violent
  du tableau, sur 25 commandes. Une ou deux commandes exceptionnelles suffisent
  à produire ce chiffre.
- **L'Irlande : 1 020 € de moyenne** — et vous savez depuis la séance 2.3
  qu'elle n'a que **deux clients**.
- **Le Royaume-Uni : moyenne 394 €, médiane 301 €.** L'écart le plus faible :
  c'est le marché le plus régulier, celui sur lequel une moyenne veut dire
  quelque chose.

> ⚠️ **Le filtre `count >= 20` n'est pas de la coquetterie.** Sans lui, le
> Japon arrive en tête du classement avec 2 178 € de moyenne… sur 12 commandes.
> Regardez toujours l'effectif avant de commenter un groupe.

In [ ]:
# Trier avant de tracer : un graphique en barres non trie est illisible
parpays.query("count >= 20").sort_values("median")["median"].plot(
    kind="barh", figsize=(7, 4))
plt.title("Mediane de ca par pays")
plt.xlabel("ca : montant de la commande (euros)")
plt.show()

## 5. Moyenne pondérée et moyenne non pondérée

Vous avez maintenant un tableau de moyennes par pays. Question naturelle :
**peut-on en déduire la moyenne de l'entreprise ?**

La réponse tient en une expérience.

In [ ]:
non_ponderee = cmd.groupby("pays")["ca"].mean().mean()   ## moyenne DES moyennes
globale = cmd["ca"].mean()                              ## sur les 1 955 commandes

print("moyenne des moyennes par pays :", round(non_ponderee, 2))
print("moyenne des commandes         :", round(globale, 2))

**750,88 € contre 589,73 €.** Deux calculs, le même fichier, 27 % d'écart.

### Ce qui se passe

Une moyenne peut donner à chaque observation un **poids** différent.

- La **moyenne non pondérée** de moyennes donne le même poids à chaque pays :
  le Canada, avec **une** commande, pèse autant que le Royaume-Uni et ses 798.
  Chacun des 23 pays compte pour 1/23 du résultat.
- La **moyenne pondérée** donne à chaque pays un poids proportionnel à son
  nombre de commandes. C'est celle-là qui répond à « quel est le panier moyen
  de l'entreprise ? ».

Vérifions-le : reprenons les moyennes par pays, et pondérons chacune par son
effectif.

In [ ]:
pp = cmd.groupby("pays")["ca"].agg(["count", "mean"])   ## effectif ET moyenne

# chaque moyenne compte a hauteur du nombre de commandes qu'elle resume
ponderee = (pp["mean"] * pp["count"]).sum() / pp["count"].sum()

print("moyenne ponderee :", round(ponderee, 2))   ## a comparer aux 589,73

**589,73 €.** Au centime près la moyenne globale — et ce n'est pas une
coïncidence : pondérer chaque moyenne de groupe par son effectif revient à
additionner tous les montants et à diviser par le nombre total de commandes.
C'est la définition de la moyenne, réécrite en deux étapes.

> 💡 `np.average(pp["mean"], weights=pp["count"])` fait la même chose en une
> ligne. Le calcul explicite ci-dessus a l'avantage de montrer *où* est la
> pondération.

### Laquelle des deux faut-il ?

Celle qui correspond à la question posée. Aucune des deux n'est fausse en
soi ; ce qui est faux, c'est de se tromper d'étiquette.

| La question | La moyenne |
|---|---|
| « Quel est le panier moyen de l'entreprise ? » | **pondérée** — chaque commande compte une fois |
| « Combien vaut le panier moyen d'un marché typique ? » | **non pondérée** — chaque pays compte une fois |

Le second calcul a un sens si vous comparez des marchés entre eux. Mais il
répond à une question sur les **pays**, pas sur les **commandes** — et 750,88 €
n'est le panier moyen de personne.

> ⚠️ **Le réflexe.** Devant une moyenne de moyennes, posez toujours la
> question : *qu'est-ce qui compte pour un, ici ?* Si la réponse n'est pas
> l'individu sur lequel porte votre question, il faut pondérer.

Ce piège ne se limite pas aux groupes. Il se pose à chaque fois qu'on divise :
un « chiffre d'affaires moyen par jour » calculé sur 7 jours n'a de sens que
si l'enseigne ouvre 7 jours. Vous vérifierez ce point sur ce fichier en
partie 2 des exercices.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| tout d'un coup | `df["ca"].describe()` |
| le centre, version fragile | `df["ca"].mean()` |
| le centre, version robuste | `df["ca"].median()` |
| un seuil, ici les 10 % du haut | `df["ca"].quantile(0.9)` |
| la dispersion, version fragile | `df["ca"].std()` |
| la dispersion, version robuste | `q3 - q1` |
| la forme | `df["ca"].plot(kind="hist", bins=40)` |
| comparer des groupes | `df.groupby("pays")["ca"].agg(["count", "mean", "median"])` |
| une moyenne pondérée | `(m * n).sum() / n.sum()`, ou `np.average(m, weights=n)` |
| combien de modalités | `df["jour"].nunique()` |

## Les trois réflexes de la séance

1. **Toujours afficher la médiane à côté de la moyenne.** L'écart entre les
   deux mesure l'asymétrie. Ici : 590 € contre 356 €, et seulement 29 % des
   commandes dépassent la « moyenne ».

2. **Une moyenne de moyennes est une moyenne _non pondérée_.** Elle donne le
   même poids à un pays qui pèse 798 commandes et à un pays qui en pèse 12.
   Pour un indicateur qui porte sur les commandes, il faut pondérer par les
   effectifs — ou plus simplement repartir des données individuelles.

3. **Compter les modalités avant de commenter un groupe.** Une moyenne sur
   12 commandes n'est pas un résultat, c'est une anecdote.